# ML06 · PyTorch 张量与 autograd

用 PyTorch 的张量（tensor）与自动微分（autograd），让框架替你算梯度，并亲手把梯度下降（gradient descent）重做一遍。

## 学习目标
- 能创建并操作 PyTorch 张量（tensor），理解它与 numpy 数组的关系与差异。
- 理解 requires_grad 如何打开对张量的梯度追踪，并创建计算图（computation graph）的直觉。
- 能调用 backward 触发反向传播（backpropagation），并从 .grad 读出梯度（gradient）。
- 看懂框架自动算出的梯度，与手写公式得到的结果一致。
- 用 autograd 重写一次梯度下降（gradient descent），体会「不用手推导数」的便利。

In [ ]:
# 概念：加载 PyTorch 并固定乱数种子，让每次运行结果一致（方便对照）
import torch

torch.manual_seed(0)   # 固定种子，后面造的假数据与随机初始化才可重现

print("PyTorch 版本:", torch.__version__)

## 张量基础：创建与运算

张量（tensor）是 PyTorch 的基本数据容器，可视为「会记得运算过程」的 numpy 数组。先把创建与运算的手感创建起来，后面的自动微分才有对象可操作。

为什么重要：深度学习的所有输入、参数、输出都是张量；它和 numpy 数组一样有形状（shape）与数据类型（dtype），但能在 GPU 上运算，并支持自动微分。

常见创建方式：`torch.tensor(数据)`、`torch.zeros(形状)`、`torch.arange(起, 迄, 步长)`。

In [ ]:
# 概念：用多种方式创建张量，并检查形状 shape 与数据类型 dtype
import torch

# 造一份假建筑数据：五栋楼的楼高（公尺）与面积（平方公尺）
heights = torch.tensor([12.0, 30.0, 9.0, 45.0, 18.0])      # 用 list 直接建成一维张量
areas = torch.tensor([85.0, 120.0, 60.0, 200.0, 95.0])

zeros = torch.zeros(5)              # 长度 5、全为 0 的张量，常用来初始化
ramp = torch.arange(0, 5, 1)        # 0,1,2,3,4；类似 numpy.arange

print("楼高:", heights)
print("面积:", areas)
print("zeros:", zeros, " arange:", ramp)
# shape 读作各轴长度；dtype 预设为 float32（list 含小数时）
print("楼高 shape:", heights.shape, " dtype:", heights.dtype)
# 注意：arange 给整数时 dtype 会是 int64，与 float 张量运算前常需转型
print("arange dtype:", ramp.dtype)

### 张量运算与 numpy 互转

张量支持逐元素（element-wise）的加减乘除与矢量化运算，写法与 numpy 几乎相同；矩阵乘法（matrix multiplication）用 `@` 或 `torch.matmul`。

张量与 numpy 数组可互转：`tensor.numpy()` 转出、`torch.from_numpy(数组)` 转入。何时用：常在数据准备阶段用 numpy，喂进模型前转成张量。

In [ ]:
# 概念：张量的逐元素运算、矩阵乘法，以及与 numpy 的互转
import torch
import numpy as np

heights = torch.tensor([12.0, 30.0, 9.0, 45.0, 18.0])
areas = torch.tensor([85.0, 120.0, 60.0, 200.0, 95.0])

# 逐元素：每栋楼高加 3 公尺（矢量化，不需循环）
print("楼高各加 3:", heights + 3)
# 逐元素相乘：可粗略当成「楼高乘面积」的量体指针
print("楼高×面积:", heights * areas)

# 矩阵乘法：把两个一维矢量做内积，得到一个纯量
dot = heights @ areas                 # @ 即 matmul；一维对一维是内积
print("内积（纯量）:", dot)

# 互转：张量 -> numpy -> 张量
as_numpy = heights.numpy()            # 转成 numpy 数组
back = torch.from_numpy(as_numpy)     # 再转回张量
print("转回张量是否一致:", torch.equal(heights, back))

## requires_grad 与计算图

把张量的 `requires_grad` 设为 True，PyTorch 就会在背后记录对它做的每一步运算，串成一张可回溯的计算图（computation graph）。这是自动微分的前提。

几个关键名词：
- 叶节点（leaf node）：我们自己创建、要对它求梯度的张量（例如参数 w、b）。
- grad_fn：由运算自动产生的张量会带一个 grad_fn，记录「它是由哪种运算算出来的」，反向时靠它回推。

形状（示意）：`w = torch.tensor(值, requires_grad=True)`。

In [ ]:
# 概念：打开 requires_grad 后，运算结果会带 grad_fn，记录计算图
import torch

# 把 w、b 当成可训练参数：requires_grad=True 才会被追踪
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)                 # 输入数据不需梯度，维持预设 False

y = w * x + b                         # 一个简单纯量运算，会被记进计算图

print("y =", y)
# 叶节点 w、b 的 is_leaf 为 True；y 是运算结果，不是叶节点
print("w 是叶节点:", w.is_leaf, " y 是叶节点:", y.is_leaf)
# grad_fn 记录 y 由哪种运算产生（这里是加法 AddBackward），反向时靠它回推
print("y.grad_fn:", y.grad_fn)
# 注意：未打开 requires_grad 的 x 没有 grad_fn，不会被追踪
print("x.requires_grad:", x.requires_grad)

## backward 与读取 .grad

自动微分（autograd）的核心动作是 `backward`：它从输出沿计算图往回走，把每个叶节点的梯度（gradient）算出来，填进该张量的 `.grad`。

两个务必记住的细节：
- 梯度累积（gradient accumulation）：`.grad` 是「累加」的，每轮训练前要用 zero_grad 清零，否则梯度会叠加导致更新错误。
- no_grad 区块：只想前向推论、不想被追踪时，用 `with torch.no_grad():` 暂停建图，省内存也避免误算梯度。

In [ ]:
# 概念：调用 backward 触发反向传播，从 .grad 读梯度，并示范清零与累积
import torch

w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)

y = w * x + b
y.backward()                          # 从 y 往回走计算图，把梯度填进 w.grad、b.grad

# dy/dw = x = 3、dy/db = 1，可手算验证
print("第一次 w.grad:", w.grad, " b.grad:", b.grad)

# 注意：不清零再算第二次，.grad 会累加（变成两次的和）
y2 = w * x + b
y2.backward()
print("未清零的第二次 w.grad:", w.grad)   # 变成 3+3=6，这通常不是我们要的

# 技巧：每轮反向前先清零，梯度才正确
w.grad.zero_()                        # 就地清零，底线结尾代表 in-place
b.grad.zero_()
y3 = w * x + b
y3.backward()
print("清零后重算 w.grad:", w.grad, " b.grad:", b.grad)

# 概念：no_grad 区块内的运算不建计算图、不追踪梯度
with torch.no_grad():
    preview = w * x + b               # 只想看结果、不需要梯度
print("no_grad 下的结果 requires_grad:", preview.requires_grad)

## 梯度的直觉：框架在算什么

梯度（gradient）告诉你「往哪个方向、调多少，能让损失（loss）下降」。对参数而言，梯度是损失对该参数的偏导数（partial derivative）。

关键极简公式：对平方误差损失 loss = (y_pred − y)^2，梯度方向由 2 (y_pred − y) 主导。
- 梯度为正：表示参数增大会让损失上升，所以应往减小的方向调。
- 梯度为负：表示参数增大会让损失下降，应往增大的方向调。

下面用一维线性关系比对：手算公式与 autograd 应得到相同的梯度。

In [ ]:
# 概念：对平方误差损失，比对手算梯度公式与 autograd 结果是否一致
import torch

# 自造一笔一维线性数据：用单一斜率 w 预测 y_pred = w * x
x = torch.tensor(4.0)
y_true = torch.tensor(10.0)           # 真实答案
w = torch.tensor(2.0, requires_grad=True)

y_pred = w * x
loss = (y_pred - y_true) ** 2         # 平方误差 squared error
loss.backward()                       # autograd 算出 dloss/dw

# 手算：dloss/dw = 2 (w*x - y) * x（链锁律），用同样的数值代入
manual_grad = 2 * (w.item() * x.item() - y_true.item()) * x.item()

print("autograd 梯度:", w.grad.item())
print("手算公式梯度:", manual_grad)
# 两者应几乎相同；浮点误差用 isclose 判断
print("两者是否一致:", torch.isclose(w.grad, torch.tensor(manual_grad)).item())
# 梯度为正代表 w 增大会让损失上升，故应把 w 往下调
print("梯度为正吗（正=该调小 w）:", w.grad.item() > 0)

## 用 autograd 重做梯度下降

梯度下降（gradient descent）的步骤不变：算损失 → 算梯度 → 沿梯度反方向更新参数。差别只在于「算梯度」这一步改用 backward 读 .grad，不必再手推导数。

更新公式：`参数 = 参数 − 学习率 × 梯度`。学习率（learning rate）控制每步走多大：太小收敛慢、太大可能发散。

循环要点：更新参数时包在 no_grad 内（更新本身不该被追踪），更新后记得 zero_grad，否则梯度会累加。

In [ ]:
# 概念：用 autograd 跑完整梯度下降，拟合一条带杂讯的线性数据
import torch

torch.manual_seed(0)

# 自造数据：真实关系 y = 3x + 2，加一点高斯杂讯让它更像真实量测
x = torch.linspace(0, 5, 30)                       # 0 到 5 取 30 个等距点
y = 3.0 * x + 2.0 + 0.5 * torch.randn(30)          # 真实斜率 3、截距 2

# 两个可训练参数，从随意初始值开始学
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.02                               # 学习率：每步更新的幅度

for step in range(200):
    y_pred = w * x + b                             # 前向：线性预测
    loss = ((y_pred - y) ** 2).mean()              # 均方误差 mean squared error
    loss.backward()                                # 反向：算 w.grad、b.grad

    # 注意：更新参数时不该被追踪，否则更新动作也会进计算图
    with torch.no_grad():
        w -= learning_rate * w.grad                # 沿梯度反方向走一步
        b -= learning_rate * b.grad
        w.grad.zero_()                             # 清零，避免下一轮梯度累加
        b.grad.zero_()

    if step % 40 == 0:                             # 每 40 步印一次损失观察下降
        print(f"step {step:3d}  loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

# 最终参数应接近真实的 w=3、b=2
print("最终 w:", round(w.item(), 3), " 最终 b:", round(b.item(), 3))

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 list 或 torch 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）楼高张量小计算（集成：张量创建 + 张量运算）
#   情境：用 list 自造五栋楼的楼高（公尺）与楼层数，想算每栋的平均层高。
#   要求：
#     1. 把楼高与楼层数各建成一个张量，检查形状 shape 与 dtype。
#     2. 用张量运算逐栋相除得到平均层高，并印出整体平均（用 .mean()）。
#   验收：应该看到一个长度为五的平均层高张量，与一个整体平均纯量。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）一个参数的梯度检查（集成：requires_grad + backward + 读 .grad + 梯度直觉）
#   情境：自造一组「容积率（floor area ratio）对土地价格」的线性数据（含少量杂讯），
#         先用单一斜率参数 w 做预测 y_pred = w * x。
#   要求：
#     1. 把 w 设为 requires_grad=True，用平方误差（squared error）算出损失。
#     2. 调用 backward，读出 w.grad。
#     3. 另用手写公式 2*(w*x - y)*x（对所有样本取平均）算同一个梯度，印出两者比对。
#   验收：应看到 autograd 与手算梯度数值几乎相同，
#         并能说出梯度正负代表 w 该往下或往上调。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）用 autograd 拟合楼高与造价（集成：张量 + 计算图 + backward + zero_grad + 梯度下降）
#   情境：自造一批楼高对造价的数据（线性趋势加杂讯），目标用 w 与 b 两个参数拟合一条直线。
#   要求：
#     1. 设置 w、b 为可训练张量（requires_grad=True），写出预测与均方误差损失。
#     2. 跑梯度下降循环：每轮 backward、读 .grad、在 no_grad 下更新参数，更新后记得 zero_grad。
#     3. 记录损失随迭代变化，并比较两种学习率（例如 0.01 与一个明显过大的值）对收敛的影响。
#   验收：应看到损失逐步下降、拟合出的 w 与 b 接近自造数据的真实趋势，
#         且过大学习率会使损失发散（变大甚至变成 nan）。
# TODO: 在这里完成


## 小结

- 张量（tensor）是 PyTorch 的基本数据容器，与 numpy 数组同源、可互转，同样有 shape 与 dtype，但能在背后记录运算以支持自动微分。
- 把张量的 requires_grad 设为 True，PyTorch 会把对它的每步运算串成计算图（computation graph）；自己创建的参数是叶节点（leaf node），运算结果带有 grad_fn。
- backward 从输出沿计算图往回走，把梯度填进叶节点的 .grad；.grad 会累积，每轮务必 zero_grad，只想推论时用 no_grad 暂停建图。
- 梯度（gradient）指出让损失下降的方向与幅度；对平方误差，方向由 2 (y_pred − y) 主导，autograd 的结果与手算公式一致。
- 用 autograd 重做梯度下降，更新公式（参数 = 参数 − 学习率 × 梯度）不变，省去手推导数的工作；学习率太大会发散、太小收敛慢。